# 03 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

your 131072x1 screen size is bogus. expect trouble
26/05/04 21:13:29 WARN Utils: Your hostname, DESKTOP-J27O5E0 resolves to a loopback address: 127.0.1.1; using 172.22.146.234 instead (on interface eth0)
26/05/04 21:13:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/william/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/william/.ivy2/cache
The jars for the packages stored in: /home/william/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-87f4f0c2-83c3-4e08-a539-477f709563e1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 369ms :: artifacts dl 23ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.ha

SparkSession criada com sucesso!


## 2. Registrar Tabelas Delta como SQL Tables

In [2]:
# Registrar as tabelas Delta Lake para uso com Spark SQL
tabelas_delta = [
    'Customers', 'Insurance_Types', 'Policies', 'Claims',
    'Payments', 'Agents', 'Agent_Policies'
]

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

# Listar tabelas registradas
print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

26/05/04 21:14:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Tabelas registradas no Spark:
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|default  |agent_policies |false      |
|default  |agents         |false      |
|default  |claims         |false      |
|default  |customers      |false      |
|default  |insurance_types|false      |
|default  |payments       |false      |
|default  |policies       |false      |
+---------+---------------+-----------+



## 3. Consultar Dados Atuais (SELECT)

In [4]:
# Visualizar as tabelas de domínio
print('=== CUSTOMERS ===')
spark.sql('SELECT * FROM Customers ORDER BY customer_id').show()

print('=== INSURANCE TYPES ===')
spark.sql('SELECT * FROM Insurance_Types ORDER BY insurance_type_id').show()

print('=== Policies (primeiros 10) ===')
spark.sql('SELECT * FROM Policies ORDER BY policy_id LIMIT 10').show()

=== CUSTOMERS ===


26/05/04 21:18:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+----------+---------+-------------+--------------------+--------+-------------+
|customer_id|first_name|last_name|date_of_birth|               email|   phone|      address|
+-----------+----------+---------+-------------+--------------------+--------+-------------+
|          1|      John|      Doe|   1985-01-10|  john.doe@email.com|555-1111|  123 Main St|
|          2|      Jane|    Smith|   1990-02-20|jane.smith@email.com|555-1112|   456 Oak St|
|          3|    Carlos|    Silva|   1978-03-15|  carlos.s@email.com|555-1113|  789 Pine St|
|          4|      Anna|    Brown|   1988-04-18|    anna.b@email.com|555-1114| 321 Maple St|
|          5|      Mike|   Wilson|   1992-05-22|    mike.w@email.com|555-1115| 654 Cedar St|
|          6|     Laura|   Garcia|   1983-06-30|   laura.g@email.com|555-1116| 987 Birch St|
|          7|      Paul|   Taylor|   1975-07-12|    paul.t@email.com|555-1117|741 Walnut St|
|          8|      Emma| Anderson|   1995-08-09|    emma.a@email.com|5

+-----------------+----------+--------------------+
|insurance_type_id| type_name|         description|
+-----------------+----------+--------------------+
|                1|      Auto|Automobile insurance|
|                2|    Health|    Health insurance|
|                3|      Life|      Life insurance|
|                4|      Home|Homeowners insurance|
|                5|    Travel|    Travel insurance|
|                6|       Pet|       Pet insurance|
|                7|    Dental|    Dental insurance|
|                8|    Vision|    Vision insurance|
|                9|  Business|  Business insurance|
|               10|Disability|Disability insurance|
+-----------------+----------+--------------------+

=== Policies (primeiros 10) ===


+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|policy_id|customer_id|insurance_type_id|policy_number|start_date|  end_date|premium_amount|policy_status|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|        1|          1|                1|      POL1001|2025-01-01|2025-12-31|        1200.0|       Active|
|        2|          2|                2|      POL1002|2025-01-01|2025-12-31|        1800.0|       Active|
|        3|          3|                3|      POL1003|2025-02-01|2026-01-31|        2500.0|       Active|
|        4|          4|                4|      POL1004|2025-02-01|2026-01-31|         900.0|       Active|
|        5|          5|                5|      POL1005|2025-03-01|2026-02-28|         600.0|       Active|
|        6|          6|                6|      POL1006|2025-03-01|2026-02-28|         400.0|       Active|
|        7|          7|              

In [5]:
# Contagem de registros por tabela
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<15} {count:>10}')

Tabela           Registros
---------------------------
Customers               10
Insurance_Types         10
Policies                10


Claims                  10


Payments                10


Agents                  10
Agent_Policies          10


---
## 4. INSERT - Inserir Novos Registros

Vamos inserir novos registros nas tabelas `Customers`, `Claims` e `Policies`.

In [ ]:
# INSERT - Novos Customers
print('--- INSERT: Novos Customers ---')
spark.sql("SELECT COUNT(*) as antes FROM Customers").show()

spark.sql("""
    INSERT INTO Customers VALUES
    (11, 'Lucas', 'Ferreira', '1991-03-12', 'lucas.ferreira@email.com', '555-1121', 'Rua das Flores, 100'),
    (12, 'Mariana', 'Souza', '1987-07-25', 'mariana.souza@email.com', '555-1122', 'Av. Brasil, 450'),
    (13, 'Rafael', 'Oliveira', '1994-11-08', 'rafael.oliveira@email.com', '555-1123', 'Rua Central, 78')
""")

spark.sql("SELECT * FROM Customers ORDER BY customer_id").show()
print('3 novos clientes inseridos!')

--- INSERT: Novos Customers ---
+-----+
|antes|
+-----+
|   10|
+-----+



+-----------+----------+---------+-------------+--------------------+--------+-------------------+
|customer_id|first_name|last_name|date_of_birth|               email|   phone|            address|
+-----------+----------+---------+-------------+--------------------+--------+-------------------+
|          1|      John|      Doe|   1985-01-10|  john.doe@email.com|555-1111|        123 Main St|
|          2|      Jane|    Smith|   1990-02-20|jane.smith@email.com|555-1112|         456 Oak St|
|          3|    Carlos|    Silva|   1978-03-15|  carlos.s@email.com|555-1113|        789 Pine St|
|          4|      Anna|    Brown|   1988-04-18|    anna.b@email.com|555-1114|       321 Maple St|
|          5|      Mike|   Wilson|   1992-05-22|    mike.w@email.com|555-1115|       654 Cedar St|
|          6|     Laura|   Garcia|   1983-06-30|   laura.g@email.com|555-1116|       987 Birch St|
|          7|      Paul|   Taylor|   1975-07-12|    paul.t@email.com|555-1117|      741 Walnut St|
|         

In [9]:
# INSERT - Novas Claims
print('--- INSERT: Novas Claims ---')

spark.sql("""
    INSERT INTO Claims VALUES
    (11, 1, '2025-06-10', 1200.00, 'Pending',  'Car accident with minor damage'),
    (12, 2, '2025-06-12', 3500.00, 'Approved', 'Hospital expenses for surgery'),
    (13, 3, '2025-06-15', 8000.00, 'Under Review', 'Life insurance beneficiary claim'),
    (14, 4, '2025-06-18', 15000.00, 'Approved', 'House damage due to flooding'),
    (15, 5, '2025-06-20', 2200.00, 'Rejected', 'Trip cancellation not covered')
""")


spark.sql("SELECT * FROM Claims WHERE claim_amount >= 100 ORDER BY claim_id").show()
print('5 novos sinistros inseridos!')

--- INSERT: Novas Claims ---


+--------+---------+----------+------------+------------+--------------------+
|claim_id|policy_id|claim_date|claim_amount|claim_status|         description|
+--------+---------+----------+------------+------------+--------------------+
|       1|        1|2025-06-01|       500.0|    Approved|        Sample claim|
|       2|        2|2025-06-01|       500.0|    Approved|        Sample claim|
|       3|        3|2025-06-01|       500.0|    Approved|        Sample claim|
|       4|        4|2025-06-01|       500.0|    Approved|        Sample claim|
|       5|        5|2025-06-01|       500.0|    Approved|        Sample claim|
|       6|        6|2025-06-01|       500.0|    Approved|        Sample claim|
|       7|        7|2025-06-01|       500.0|    Approved|        Sample claim|
|       8|        8|2025-06-01|       500.0|    Approved|        Sample claim|
|       9|        9|2025-06-01|       500.0|    Approved|        Sample claim|
|      10|       10|2025-06-01|       500.0|    Appr

In [11]:
# INSERT - Novas Policies
print('--- INSERT: Novas Policies ---')

spark.sql("""
    INSERT INTO Policies VALUES
    (11, 11, 1, 'POL1011', '2025-06-01', '2026-05-31', 1450.00, 'Active'),
    (12, 12, 2, 'POL1012', '2025-06-05', '2026-06-04', 2100.00, 'Active')
""")

spark.sql("""
    SELECT *
    FROM Policies
    WHERE start_date >= add_months(current_date(), -12)
""").show()
print('2 novas Apólices inseridas!')

--- INSERT: Novas Policies ---


+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|policy_id|customer_id|insurance_type_id|policy_number|start_date|  end_date|premium_amount|policy_status|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|       11|         11|                1|      POL1011|2025-06-01|2026-05-31|        1450.0|       Active|
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2100.0|       Active|
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2100.0|       Active|
|       11|         11|                1|      POL1011|2025-06-01|2026-05-31|        1450.0|       Active|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+

2 novas Apólices inseridas!


---
## 5. UPDATE - Atualizar Registros

Vamos atualizar registros existentes nas tabelas.

In [12]:
# UPDATE - Atualizar Customer
print('--- UPDATE: Atualizar Customer ---')

print('ANTES:')
spark.sql("""
    SELECT *
    FROM Customers
    WHERE customer_id = 11
""").show()

spark.sql("""
    UPDATE Customers
    SET
        first_name = 'Lucas Henrique',
        email = 'lucas.h.ferreira@email.com'
    WHERE customer_id = 11
""")

print('DEPOIS:')
spark.sql("""
    SELECT *
    FROM Customers
    WHERE customer_id = 11
""").show()

print('Customer atualizado com sucesso!')

--- UPDATE: Atualizar Customer ---
ANTES:
+-----------+----------+---------+-------------+--------------------+--------+-------------------+
|customer_id|first_name|last_name|date_of_birth|               email|   phone|            address|
+-----------+----------+---------+-------------+--------------------+--------+-------------------+
|         11|     Lucas| Ferreira|   1991-03-12|lucas.ferreira@em...|555-1121|Rua das Flores, 100|
+-----------+----------+---------+-------------+--------------------+--------+-------------------+

DEPOIS:
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|customer_id|    first_name|last_name|date_of_birth|               email|   phone|            address|
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|         11|Lucas Henrique| Ferreira|   1991-03-12|lucas.h.ferreira@...|555-1121|Rua das Flores, 100|
+-----------+--------------+---------+----

In [ ]:
# UPDATE - Atualizar Claim (Sinistro)
print('--- UPDATE: Atualizar Claim ---')

print('ANTES:')
spark.sql("""
    SELECT *
    FROM Claims
    WHERE claim_id = 11
""").show()

spark.sql("""
    UPDATE Claims
    SET
        claim_status = 'Approved',
        claim_amount = 1350.00,
        description = 'Car accident approved after inspection'
    WHERE claim_id = 11
""")

print('DEPOIS:')
spark.sql("""
    SELECT *
    FROM Claims
    WHERE claim_id = 11
""").show()

print('Claim atualizado com sucesso!')


--- UPDATE: Atualizar Claim ---
ANTES:
+--------+---------+----------+------------+------------+--------------------+
|claim_id|policy_id|claim_date|claim_amount|claim_status|         description|
+--------+---------+----------+------------+------------+--------------------+
|      11|        1|2025-06-10|      1200.0|     Pending|Car accident with...|
|      11|        1|2025-06-10|      1200.0|     Pending|Car accident with...|
+--------+---------+----------+------------+------------+--------------------+



DEPOIS:


+--------+---------+----------+------------+------------+--------------------+
|claim_id|policy_id|claim_date|claim_amount|claim_status|         description|
+--------+---------+----------+------------+------------+--------------------+
|      11|        1|2025-06-10|      1350.0|    Approved|Car accident appr...|
|      11|        1|2025-06-10|      1350.0|    Approved|Car accident appr...|
+--------+---------+----------+------------+------------+--------------------+

Claim atualizado com sucesso!


In [14]:
# UPDATE com DeltaTable API (alternativa ao SQL)
print('--- UPDATE via DeltaTable API ---')
from pyspark.sql.functions import lit


dt_policies = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/Policies')

dt_policies.update(
    condition="policy_id = 12",
    set={
        "policy_status": lit("Renewed"),
        "premium_amount": lit(2300.00)
    }
)

spark.sql("SELECT * FROM Policies WHERE policy_id = 12").show()


--- UPDATE via DeltaTable API ---
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|policy_id|customer_id|insurance_type_id|policy_number|start_date|  end_date|premium_amount|policy_status|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2300.0|      Renewed|
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2300.0|      Renewed|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+



---
## 6. DELETE - Remover Registros

Vamos deletar registros de tabelas Delta.

In [17]:
# DELETE - Customers via SQL
print('--- DELETE: Remover Customers teste ---')

print('ANTES:')
spark.sql("SELECT * FROM Customers WHERE customer_id = 13").show()

spark.sql("""
    DELETE FROM Customers
    WHERE customer_id = 13
""")

print('DEPOIS:')
spark.sql("SELECT * FROM Customers WHERE customer_id = 13").show()

--- DELETE: Remover Customers teste ---
ANTES:
+-----------+----------+---------+-------------+-----+-----+-------+
|customer_id|first_name|last_name|date_of_birth|email|phone|address|
+-----------+----------+---------+-------------+-----+-----+-------+
+-----------+----------+---------+-------------+-----+-----+-------+

DEPOIS:
+-----------+----------+---------+-------------+-----+-----+-------+
|customer_id|first_name|last_name|date_of_birth|email|phone|address|
+-----------+----------+---------+-------------+-----+-----+-------+
+-----------+----------+---------+-------------+-----+-----+-------+



In [18]:
# DELETE - Claims via SQL
print('--- DELETE: Remover Claims teste ---')

print('ANTES:')
spark.sql("SELECT * FROM Claims WHERE claim_id >= 10").show()

spark.sql("""
    DELETE FROM Claims
    WHERE claim_id = 15
""")

print('DEPOIS:')
spark.sql("SELECT * FROM Claims WHERE claim_id >= 10").show()

--- DELETE: Remover Claims teste ---
ANTES:
+--------+---------+----------+------------+------------+--------------------+
|claim_id|policy_id|claim_date|claim_amount|claim_status|         description|
+--------+---------+----------+------------+------------+--------------------+
|      10|       10|2025-06-01|       500.0|    Approved|        Sample claim|
|      11|        1|2025-06-10|      1350.0|    Approved|Car accident appr...|
|      11|        1|2025-06-10|      1350.0|    Approved|Car accident appr...|
|      13|        3|2025-06-15|      8000.0|Under Review|Life insurance be...|
|      13|        3|2025-06-15|      8000.0|Under Review|Life insurance be...|
|      15|        5|2025-06-20|      2200.0|    Rejected|Trip cancellation...|
|      12|        2|2025-06-12|      3500.0|    Approved|Hospital expenses...|
|      12|        2|2025-06-12|      3500.0|    Approved|Hospital expenses...|
|      15|        5|2025-06-20|      2200.0|    Rejected|Trip cancellation...|
|      1

In [19]:
# DELETE - Policies via DeltaTable API
print('--- DELETE via DeltaTable API ---')
from delta.tables import DeltaTable

dt_policies = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/Policies')

print('ANTES:')
spark.sql("SELECT * FROM Policies WHERE policy_id >= 10").show()

dt_policies.delete("policy_id = 12")

print('DEPOIS:')
spark.sql("SELECT * FROM Policies WHERE policy_id >= 10").show()

--- DELETE via DeltaTable API ---
ANTES:
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|policy_id|customer_id|insurance_type_id|policy_number|start_date|  end_date|premium_amount|policy_status|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-------------+
|       10|         10|               10|      POL1010|2025-05-01|2026-04-30|        2200.0|       Active|
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2300.0|      Renewed|
|       12|         12|                2|      POL1012|2025-06-05|2026-06-04|        2300.0|      Renewed|
|       11|         11|                1|      POL1011|2025-06-01|2026-05-31|        1450.0|       Active|
|       11|         11|                1|      POL1011|2025-06-01|2026-05-31|        1450.0|       Active|
+---------+-----------+-----------------+-------------+----------+----------+--------------+-----------

---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações.

In [20]:
# Histórico da tabela Policies
print('=== HISTORICO DA TABELA POLICIES ===')
dt_policies = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/Policies')
dt_policies.history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

=== HISTORICO DA TABELA POLICIES ===
+-------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
# Histórico da tabela Claims
print('=== HISTORICO DA TABELA CLAIMS ===')
dt_claims = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/Claims')
dt_claims.history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

=== HISTORICO DA TABELA CLAIMS ===
+-------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores dos dados.

In [22]:
# Versão atual da tabela Customers
print('=== CUSTOMERS - VERSAO ATUAL ===')
spark.sql('SELECT * FROM Customers ORDER BY customer_id').show()

=== CUSTOMERS - VERSAO ATUAL ===
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|customer_id|    first_name|last_name|date_of_birth|               email|   phone|            address|
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|          1|          John|      Doe|   1985-01-10|  john.doe@email.com|555-1111|        123 Main St|
|          2|          Jane|    Smith|   1990-02-20|jane.smith@email.com|555-1112|         456 Oak St|
|          3|        Carlos|    Silva|   1978-03-15|  carlos.s@email.com|555-1113|        789 Pine St|
|          4|          Anna|    Brown|   1988-04-18|    anna.b@email.com|555-1114|       321 Maple St|
|          5|          Mike|   Wilson|   1992-05-22|    mike.w@email.com|555-1115|       654 Cedar St|
|          6|         Laura|   Garcia|   1983-06-30|   laura.g@email.com|555-1116|       987 Birch St|
|          7|          Paul|   Taylor|  

In [23]:
# Time Travel - Ler versão 0 (estado original)
print('=== CUSTOMERS - VERSAO 0 (ORIGINAL) ===')
df_v0 = spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .load(f's3a://{BRONZE_BUCKET}/Customers')

df_v0.orderBy('customer_id').show()

=== CUSTOMERS - VERSAO 0 (ORIGINAL) ===


+-----------+----------+---------+-------------+--------------------+--------+-------------+
|customer_id|first_name|last_name|date_of_birth|               email|   phone|      address|
+-----------+----------+---------+-------------+--------------------+--------+-------------+
|          1|      John|      Doe|   1985-01-10|  john.doe@email.com|555-1111|  123 Main St|
|          2|      Jane|    Smith|   1990-02-20|jane.smith@email.com|555-1112|   456 Oak St|
|          3|    Carlos|    Silva|   1978-03-15|  carlos.s@email.com|555-1113|  789 Pine St|
|          4|      Anna|    Brown|   1988-04-18|    anna.b@email.com|555-1114| 321 Maple St|
|          5|      Mike|   Wilson|   1992-05-22|    mike.w@email.com|555-1115| 654 Cedar St|
|          6|     Laura|   Garcia|   1983-06-30|   laura.g@email.com|555-1116| 987 Birch St|
|          7|      Paul|   Taylor|   1975-07-12|    paul.t@email.com|555-1117|741 Walnut St|
|          8|      Emma| Anderson|   1995-08-09|    emma.a@email.com|5

In [24]:
# Time Travel - Comparar versão original vs atual
print('=== COMPARACAO: CUSTOMERS (Versao 0 vs Atual) ===')

df_original = spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .load(f's3a://{BRONZE_BUCKET}/Customers')

df_atual = spark.read.format('delta') \
    .load(f's3a://{BRONZE_BUCKET}/Customers')

print(f'Versao 0: {df_original.count()} registros')
print(f'Atual:    {df_atual.count()} registros')

print('\nRegistros ADICIONADOS:')
df_atual.subtract(df_original).show()

=== COMPARACAO: CUSTOMERS (Versao 0 vs Atual) ===


Versao 0: 10 registros
Atual:    12 registros

Registros ADICIONADOS:
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|customer_id|    first_name|last_name|date_of_birth|               email|   phone|            address|
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+
|         11|Lucas Henrique| Ferreira|   1991-03-12|lucas.h.ferreira@...|555-1121|Rua das Flores, 100|
|         12|       Mariana|    Souza|   1987-07-25|mariana.souza@ema...|555-1122|    Av. Brasil, 450|
+-----------+--------------+---------+-------------+--------------------+--------+-------------------+



---
## 9. Resumo Final

In [25]:
print('=' * 60)
print('RESUMO DAS OPERACOES DML REALIZADAS')
print('=' * 60)
print()
print('INSERT:')
print('  - 3 novos customers (Lucas, Mariana, Rafael)')
print('  - 5 novos claims (Pending, Approved, Under Review, etc.)')
print('  - 2 novas policies (Auto e Health)')
print()
print('UPDATE:')
print('  - customer 11 nome e email atualizados')
print('  - claim 11 status e valor atualizados')
print('  - policy 12 status e premium atualizados (via DeltaTable API)')
print()
print('DELETE:')
print('  - customer 13 removido')
print('  - claim 15 removido')
print('  - policy 12 removida (via DeltaTable API)')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Historico completo das tabelas Customers, Claims e Policies')
print('  - Leitura de versoes anteriores (versionAsOf)')
print('  - Comparacao entre versao original e atual')
print('=' * 60)

RESUMO DAS OPERACOES DML REALIZADAS

INSERT:
  - 3 novos customers (Lucas, Mariana, Rafael)
  - 5 novos claims (Pending, Approved, Under Review, etc.)
  - 2 novas policies (Auto e Health)

UPDATE:
  - customer 11 nome e email atualizados
  - claim 11 status e valor atualizados
  - policy 12 status e premium atualizados (via DeltaTable API)

DELETE:
  - customer 13 removido
  - claim 15 removido
  - policy 12 removida (via DeltaTable API)

HISTORY e TIME TRAVEL:
  - Historico completo das tabelas Customers, Claims e Policies
  - Leitura de versoes anteriores (versionAsOf)
  - Comparacao entre versao original e atual


In [26]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
